# Peri-Baseline_ON Activity Visualization by Outcome (Batch Pulse Response Analysis)

This notebook visualizes peri-Baseline_ON activity of responsive clusters for different trial outcomes, using batch analysis results and session data. It supports both single-session and batch/cross-session visualization, enabling comparison of peri-event activity (e.g., mean ± SEM PSTH, heatmaps) grouped by outcome.

**Workflow:**
- Load responsive clusters for a session (from batch analysis results)
- Load corresponding session data (e.g., from .mat or .pkl files)
- Extract trial outcomes and align cluster activity to Baseline_ON for each outcome
- Visualize peri-event activity grouped by outcome
- (Optional) Batch/cross-session visualization


## 1. Import Required Libraries
Import libraries for file handling, data analysis, and visualization.

In [1]:
import os
import glob
import pickle
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.io import loadmat
from collections import defaultdict

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.dpi'] = 100


## 2. Utility Function to Load Analysis Results
Define a function to load pulse analysis results from a .pkl file based on session ID.

In [2]:
def load_analysis_results_from_file(pkl_path):
    """
    Load pulse analysis results from a pickle file.
    Args:
        pkl_path (str): Path to the pickle file.
    Returns:
        dict: Dictionary containing analysis results.
    """
    with open(pkl_path, 'rb') as f:
        results = pickle.load(f)
    return results


## 3. Find All Session Files and Extract Session IDs
Locate all relevant `.pkl` files in the specified directory, extract session IDs and dates, and sort them by date.

In [3]:
# Directory containing .pkl files
results_dir = '../pkls'  # Adjust if needed
pkl_files = sorted(glob.glob(os.path.join(results_dir, 'fast_slow_results_*.pkl')))

# Extract session IDs and dates
sessions = []
for pkl_path in pkl_files:
    fname = os.path.basename(pkl_path)
    session_id = fname.replace('fast_slow_results_', '').replace('.pkl', '')
    # For your filenames, the date is always the last 8 digits of the session_id
    date_str = session_id[-8:] if session_id[-8:].isdigit() else None
    session_date = datetime.strptime(date_str, '%d%m%Y') if date_str else None
    sessions.append({'session_id': session_id, 'pkl_path': pkl_path, 'date': session_date})

# Sort sessions by date
sessions = sorted(sessions, key=lambda x: x['date'] if x['date'] else datetime.min)
print(f"Found {len(sessions)} session files.")


Found 12 session files.


In [4]:
sessions

[{'session_id': 'BG_031_28012025',
  'pkl_path': '../pkls\\fast_slow_results_BG_031_28012025.pkl',
  'date': datetime.datetime(2025, 1, 28, 0, 0)},
 {'session_id': 'BG_031_18022025',
  'pkl_path': '../pkls\\fast_slow_results_BG_031_18022025.pkl',
  'date': datetime.datetime(2025, 2, 18, 0, 0)},
 {'session_id': 'BG_031_25022025',
  'pkl_path': '../pkls\\fast_slow_results_BG_031_25022025.pkl',
  'date': datetime.datetime(2025, 2, 25, 0, 0)},
 {'session_id': 'BG_031_27022025',
  'pkl_path': '../pkls\\fast_slow_results_BG_031_27022025.pkl',
  'date': datetime.datetime(2025, 2, 27, 0, 0)},
 {'session_id': 'BG_031_11032025',
  'pkl_path': '../pkls\\fast_slow_results_BG_031_11032025.pkl',
  'date': datetime.datetime(2025, 3, 11, 0, 0)},
 {'session_id': 'BG_031_14032025',
  'pkl_path': '../pkls\\fast_slow_results_BG_031_14032025.pkl',
  'date': datetime.datetime(2025, 3, 14, 0, 0)},
 {'session_id': 'BG_031_26032025',
  'pkl_path': '../pkls\\fast_slow_results_BG_031_26032025.pkl',
  'date': dat

## 4. Define Analysis Pipeline Functions
Implement functions for cluster selection, filtering, responsivity analysis, and summary calculation.

In [5]:
def select_clusters(z_fast, t_vec, threshold=4.0, window=[0, 0.5]):
    window_mask = (t_vec >= window[0]) & (t_vec < window[1])
    return [clu for clu, z in z_fast.items() if np.max(z[window_mask]) > threshold]

def filter_pre_pulse(z_fast, clusters, t_vec, pre_window=[-0.05, 0], pre_activity_threshold=3):
    pre_mask = (t_vec >= pre_window[0]) & (t_vec < pre_window[1])
    filtered = []
    for clu in clusters:
        pre_max = np.max(np.abs(z_fast[clu][pre_mask]))
        if pre_max < pre_activity_threshold:
            filtered.append(clu)
    return filtered

def calculate_z_difference(z_fast, z_slow):
    return {clu: np.abs(z_fast[clu] - z_slow[clu]) for clu in z_fast.keys()}

def filter_post_pulse(z_diff, clusters, t_vec, post_window=[0, 0.5], responsivity_threshold=4):
    post_mask = (t_vec >= post_window[0]) & (t_vec < post_window[1])
    final = []
    for clu in clusters:
        max_resp = np.max(z_diff[clu][post_mask])
        if max_resp > responsivity_threshold:
            final.append(clu)
    return final

def calculate_mean_sem(z_diff, clusters):
    if not clusters:
        return None, None
    arr = np.stack([z_diff[clu] for clu in clusters])
    mean_diff = np.mean(arr, axis=0)
    sem_diff = np.std(arr, axis=0, ddof=1) / np.sqrt(arr.shape[0])
    return mean_diff, sem_diff


## 5. Visualize Peri-Baseline_ON Activity by Outcome
For a selected session, load responsive clusters, extract trial outcomes, align cluster activity to Baseline_ON, and visualize peri-event activity (mean ± SEM PSTH, heatmaps) grouped by outcome.

In [11]:
# --- Select a session to visualize ---
# You can change this index to select a different session
session_idx = 7
sess = sessions[session_idx]
print(f"Selected session: {sess['session_id']}")

# Load batch analysis results for this session
summary_pkl = os.path.join(results_dir, f'summary_{sess["session_id"]}.pkl')
with open(summary_pkl, 'rb') as f:
    summary = pickle.load(f)
responsive_clusters = summary['selected_clusters']
print(f"Number of responsive clusters: {len(responsive_clusters)}")


Selected session: BG_031_02042025
Number of responsive clusters: 25


In [12]:
os.getcwd()

'e:\\python_analysis\\git_repos\\vis_detect_analysis_Apr2023\\scripts'

In [13]:
#set current directory to e:\\python_analysis\\git_repos\\vis_detect_analysis_Apr2023
os.chdir('e:\\python_analysis\\git_repos\\vis_detect_analysis_Apr2023\pkls')

In [ ]:
# --- Load session data (adjust path and loader as needed) ---
# Example assumes .pkl file with trial info and cluster activity
# If your data is in .mat format, use scipy.io.loadmat instead

session_data_dir = results_dir  # Use the same directory as results_dir
possible_pkl = os.path.join(session_data_dir, f'session_data_{sess["session_id"]}.pkl')
possible_mat = os.path.join(session_data_dir, f'session_data_{sess["session_id"]}.mat')

if os.path.exists(possible_pkl):
    with open(possible_pkl, 'rb') as f:
        session_data = pickle.load(f)
    print(f"Loaded session data from {possible_pkl}")
elif os.path.exists(possible_mat):
    session_data = loadmat(possible_mat)
    print(f"Loaded session data from {possible_mat}")
else:
    raise FileNotFoundError(f"No session data found for {sess['session_id']}")


FileNotFoundError: No session data found for BG_031_02042025

In [20]:
# --- Extract trial outcomes and align cluster activity to Baseline_ON ---
# Assumes session_data['trialsdata'] is a list of dicts with 'trialoutcome' and 'Baseline_ON' event time
# Assumes session_data['cluster_activity'] is a dict: cluster_id -> (n_trials, n_timepoints) array
# Assumes session_data['t_vec'] is peri-event time vector

trials = session_data['trialsdata']  # list of dicts: each dict contains trial info
cluster_activity = session_data['cluster_activity']  # dict: clu -> (n_trials, n_timepoints)
t_vec = session_data['t_vec']

# Group trial indices by outcome
groups = defaultdict(list)
for i, trial in enumerate(trials):
    outcome = trial['trialoutcome']
    groups[outcome].append(i)

print(f"Found outcomes: {list(groups.keys())}")


KeyError: 'trialsdata'

In [ ]:
# --- Compute and plot peri-Baseline_ON PSTH for responsive clusters, grouped by outcome ---

fig, axes = plt.subplots(len(groups), 1, figsize=(8, 3 * len(groups)), sharex=True)
if len(groups) == 1:
    axes = [axes]

for ax, (outcome, trial_idxs) in zip(axes, groups.items()):
    # Stack all responsive cluster activity for these trials
    all_psth = []
    for clu in responsive_clusters:
        clu_data = cluster_activity[clu][trial_idxs, :]  # shape: (n_trials, n_timepoints)
        all_psth.append(clu_data)
    all_psth = np.concatenate(all_psth, axis=0)  # shape: (n_clusters * n_trials, n_timepoints)
    mean_psth = np.mean(all_psth, axis=0)
    sem_psth = np.std(all_psth, axis=0, ddof=1) / np.sqrt(all_psth.shape[0])
    ax.plot(t_vec, mean_psth, label=f'{outcome} (mean)')
    ax.fill_between(t_vec, mean_psth - sem_psth, mean_psth + sem_psth, alpha=0.3)
    ax.set_title(f'Outcome: {outcome}')
    ax.set_ylabel('Activity (a.u.)')
    ax.legend()
ax.set_xlabel('Time from Baseline_ON (s)')
plt.tight_layout()
plt.show()


In [ ]:
# --- Plot heatmaps of peri-Baseline_ON activity for all responsive clusters, grouped by outcome ---

fig, axes = plt.subplots(len(groups), 1, figsize=(10, 2.5 * len(groups)), sharex=True)
if len(groups) == 1:
    axes = [axes]

for ax, (outcome, trial_idxs) in zip(axes, groups.items()):
    # For each cluster, average across trials for this outcome
    heatmap = []
    for clu in responsive_clusters:
        clu_data = cluster_activity[clu][trial_idxs, :]  # (n_trials, n_timepoints)
        mean_clu = np.mean(clu_data, axis=0)
        heatmap.append(mean_clu)
    heatmap = np.array(heatmap)  # (n_clusters, n_timepoints)
    im = ax.imshow(heatmap, aspect='auto', extent=[t_vec[0], t_vec[-1], 0, len(responsive_clusters)],
                  cmap='viridis', origin='lower')
    ax.set_title(f'Outcome: {outcome}')
    ax.set_ylabel('Cluster #')
    fig.colorbar(im, ax=ax, orientation='vertical', label='Activity (a.u.)')
ax.set_xlabel('Time from Baseline_ON (s)')
plt.tight_layout()
plt.show()


## 6. (Optional) Batch/Cross-Session Visualization
You can extend the above code to loop over all sessions and aggregate peri-Baseline_ON activity by outcome for cross-session comparison. For each session, repeat the above steps and collect results for group-level analysis.